# Phase 3 — Step 2 (rebuild): L5 IT vs L5 ET, main experiment

**Why this notebook is the rebuild.** The v1 notebook averaged A1 / C1 / D1 features across the hashes of each neuron — exactly the operation `WORKFLOW.md §3.3` forbids ("across-hash averaging is destructive… **forbidden as a feature-engineering step in this project**"). All A1+B / A1+B+C1 / A1+B+C1+D1 / G+B+C1 numbers from v1 are invalid as headlines.

**This rebuild uses the WORKFLOW §3.6 mandatory protocol** that Phase 1 stage 2/3/4 followed: long-row training tables (one row per `(nucleus_id, condition_hash)`), `GroupKFold` by `nucleus_id`, predict per-row probabilities, then mean-aggregate to per-neuron and argmax. The neuron-level metric helper is `src/eval/metrics.py:neuron_level_score`.

**The design choices are unchanged.** Five feature blocks (G; A1+B; A1+B+C1; A1+B+C1+D1; G+B+C1), two models (LR + HGB) with `class_weight='balanced'`, four confound baselines (majority / scan-only / cc_abs-only / cc_abs-residualized winner), the same population definition, the same LOSO validity rule (deferred to notebook 03).

**Population.** L5 IT/ET = `subtype ∈ {5P-IT, 5P-ET}` on the Phase-1 working population (`units_working.parquet`, V1 / excitatory / oracle-matched / best-only, L1 dropped — 1,489 cells, 11 scans).


## 1. Setup

In [1]:
from __future__ import annotations

import sys, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.config import (PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR,
                        PROCESSED_RESULTS_DIR, RANDOM_SEED, ensure_dirs)
from src.data.loaders import build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.features.tier_b import B_FEATURE_NAMES
from src.features.tier_c import C1_FEATURE_NAMES
from src.features.tier_d import D_FEATURE_NAMES

ensure_dirs()
np.random.seed(RANDOM_SEED)

DATA_TABLES   = REPO_ROOT / 'data' / 'processed' / 'tables'
DATA_FEATURES = REPO_ROOT / 'data' / 'processed' / 'features'
DATA_RESULTS  = REPO_ROOT / 'data' / 'processed' / 'results'

UNITS_SUBTYPE_PATH = DATA_TABLES / 'units_v1_exc_best_subtype.parquet'
RESULTS_OUT  = DATA_RESULTS / 'phase3_l5_it_et_runs.parquet'
WINNER_OUT   = DATA_RESULTS / 'phase3_l5_it_et_winner.json'

CLASSES = np.array(['5P-IT', '5P-ET'])  # IT first, ET second (minority)
N_FOLDS = 5

print('Tier B features :', list(B_FEATURE_NAMES))
print('Tier C1 features:', list(C1_FEATURE_NAMES))
print('Tier D features :', list(D_FEATURE_NAMES))


Tier B features : ['b_amp_var', 'b_amp_std', 'b_amp_cv', 'b_amp_fano', 'b_amp_snr', 'b_trace_corr_mean', 'b_oracle_corr']
Tier C1 features: ['c1_rmi', 'c1_pupil_resp_slope', 'c1_state_rel_diff', 'c1_arousal_gain_diff']
Tier D features : ['d_luminance_mean', 'd_luminance_std', 'd_contrast', 'd_motion_energy', 'd_sf_low', 'd_sf_mid', 'd_sf_high', 'd_tf_low', 'd_tf_mid', 'd_tf_high']


## 2. Build the long-row modelling table (A1 + B + C1 + D)

This is the Phase 1 stage 4 join pattern, lifted verbatim:

1. `build_modeling_table(level='A1', blocks=['amp','shape'])` → long-row A1 table with one row per `(nucleus_id, condition_hash)`.
2. inner-join `B_per_hash` on `(nucleus_id, condition_hash)` — restricts to repeated hashes (n_trials ≥ 2).
3. left-join `C1_per_hash` on `(nucleus_id, condition_hash)`.
4. left-join `D_per_hash` on `condition_hash` only (D is per-stimulus).
5. broadcast `G_per_neuron` columns onto every long row of the neuron via `nucleus_id` (used by the G-containing blocks only).


In [2]:
# 2.1 A1 long
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp', 'shape'],
    label='celltype_label',  # Phase 3: predict subtype, not layer
)
print(f'A1 long table: {df_a1.shape}')

# 2.2 + B (inner join restricts to repeated hashes)
b_hash = pd.read_parquet(
    DATA_FEATURES / 'B_per_hash.parquet',
    columns=['nucleus_id', 'condition_hash'] + list(B_FEATURE_NAMES))
df_a1b = df_a1.merge(b_hash, on=['nucleus_id', 'condition_hash'],
                     how='inner', validate='one_to_one')
print(f'A1 ∩ B long table: {df_a1b.shape}')

# 2.3 + C1
c1 = pd.read_parquet(
    DATA_FEATURES / 'C1_per_hash.parquet',
    columns=['nucleus_id', 'condition_hash'] + list(C1_FEATURE_NAMES))
df_a1bc1 = df_a1b.merge(c1, on=['nucleus_id', 'condition_hash'],
                        how='left', validate='one_to_one')

# 2.4 + D
d = pd.read_parquet(
    DATA_FEATURES / 'D_per_hash.parquet',
    columns=['condition_hash'] + list(D_FEATURE_NAMES))
df_full = df_a1bc1.merge(d, on='condition_hash',
                         how='left', validate='many_to_one')
print(f'A1 ∩ B + C1 + D long table: {df_full.shape}')

# 2.5 + G broadcast (per-neuron columns attached to every row of that neuron)
G = pd.read_parquet(DATA_FEATURES / 'G_per_neuron.parquet')
g_cols = [c for c in G.columns if c.startswith('g_')]
df_full = df_full.merge(G[['nucleus_id'] + g_cols], on='nucleus_id',
                        how='left', validate='many_to_one')
print(f'+G broadcast: {df_full.shape}  ({len(g_cols)} G features per row)')


A1 long table: (2490600, 31)
A1 ∩ B long table: (1209720, 38)
A1 ∩ B + C1 + D long table: (1209720, 52)
+G broadcast: (1209720, 168)  (116 G features per row)


## 3. Restrict to L5 IT vs L5 ET, refresh fold assignment stratified by subtype

In [3]:
# Filter to L5 IT/ET cells. units_working.celltype_label is the AIBS metamodel
# call (same column as cell_type, just renamed in nb 02).
mask = df_full['celltype_label'].isin(['5P-IT', '5P-ET'])
df = df_full[mask].copy().reset_index(drop=True)
print(f'L5 IT/ET long-row table: {df.shape}, {df["nucleus_id"].nunique()} neurons, '
      f'{df["session_key"].nunique()} scans')
print('class counts (rows):'); print(df['celltype_label'].value_counts().to_string())
print('class counts (neurons):'); print(df.drop_duplicates('nucleus_id')['celltype_label'].value_counts().to_string())

# Refresh GKF fold assignments. Phase 1's precomputed gkf_fold was stratified
# by layer; for Phase 3 we re-stratify by subtype on the per-neuron table,
# then broadcast onto the long rows.
neuron_table = (df[['nucleus_id', 'celltype_label', 'session_key']]
                .drop_duplicates('nucleus_id').reset_index(drop=True))
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
neuron_table['gkf_fold_phase3'] = -1
for k, (_, te) in enumerate(sgkf.split(np.zeros((len(neuron_table), 1)),
                                       neuron_table['celltype_label'].to_numpy(),
                                       neuron_table['nucleus_id'].to_numpy())):
    neuron_table.loc[te, 'gkf_fold_phase3'] = k
df = df.merge(neuron_table[['nucleus_id', 'gkf_fold_phase3']],
              on='nucleus_id', how='left', validate='many_to_one')
print('per-fold neuron counts:')
print(neuron_table.groupby(['gkf_fold_phase3', 'celltype_label']).size().unstack(fill_value=0).to_string())


L5 IT/ET long-row table: (202504, 168), 1489 neurons, 11 scans
class counts (rows):
celltype_label
5P-IT    158984
5P-ET     43520
class counts (neurons):
celltype_label
5P-IT    1169
5P-ET     320
per-fold neuron counts:
celltype_label   5P-ET  5P-IT
gkf_fold_phase3              
0                   66    232
1                   67    231
2                   69    229
3                   60    238
4                   58    239


## 4. Define feature blocks (column lists in the long-row table)

In [4]:
amp_cols   = [c for c in df.columns if c.startswith('amp_')]
shape_cols = [c for c in df.columns if c.startswith('shape_')]
a1_cols    = amp_cols + shape_cols
b_cols     = list(B_FEATURE_NAMES)
c1_cols    = list(C1_FEATURE_NAMES)
d_cols     = list(D_FEATURE_NAMES)

BLOCKS = {
    'G':            list(g_cols),
    'A1+B':         a1_cols + b_cols,
    'A1+B+C1':      a1_cols + b_cols + c1_cols,
    'A1+B+C1+D1':   a1_cols + b_cols + c1_cols + d_cols,
    'G+B+C1':       list(g_cols) + b_cols + c1_cols,
}
for name, cols in BLOCKS.items():
    print(f'  {name:<12s} -> {len(cols):3d} features')
print()
# Check NaN counts in each block (HGB tolerates them; LR needs imputation).
for name, cols in BLOCKS.items():
    n_nan = df[cols].isna().sum().sum()
    print(f'  {name:<12s} NaN cells in long-row table: {n_nan:,}')


  G            -> 116 features
  A1+B         ->  27 features
  A1+B+C1      ->  31 features
  A1+B+C1+D1   ->  41 features
  G+B+C1       -> 127 features

  G            NaN cells in long-row table: 0
  A1+B         NaN cells in long-row table: 168,266
  A1+B+C1      NaN cells in long-row table: 715,149
  A1+B+C1+D1   NaN cells in long-row table: 715,149
  G+B+C1       NaN cells in long-row table: 623,801


## 5. CV helpers — neuron-level scoring per WORKFLOW §6.3

In [5]:
def make_pipeline(model_name: str) -> Pipeline:
    if model_name == 'LogReg':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  RobustScaler()),
            ('clf',    LogisticRegression(
                penalty='l2', C=1.0, solver='lbfgs', max_iter=400,
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ])
    if model_name == 'HGB':
        return Pipeline([
            ('clf', HistGradientBoostingClassifier(
                max_iter=100, max_depth=8, learning_rate=0.05,
                l2_regularization=1.0,
                early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_SEED)),
        ])
    raise ValueError(model_name)


def cv_run_long(X: np.ndarray, y_row: np.ndarray, groups_row: np.ndarray,
                folds_row: np.ndarray, model_name: str,
                classes=CLASSES) -> dict:
    # Long-row 5-fold CV. Returns mean and std neuron-level metrics across folds.
    fold_scores = []
    for k in range(N_FOLDS):
        tr = folds_row != k; te = folds_row == k
        if tr.sum() == 0 or te.sum() == 0: continue
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y_row[tr])
        if model_name == 'LogReg':
            pipe.fit(X[tr], y_row[tr], clf__sample_weight=sw)
        else:
            pipe.fit(X[tr], y_row[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(X[te])
        # neuron-level score: mean of per-row probs per neuron, then argmax
        sc = neuron_level_score(y_row[te], proba, groups_row[te], pipe.classes_)
        fold_scores.append(sc)
    return summarize_cv_runs(fold_scores)


def cv_run_long_residualized(X: np.ndarray, y_row: np.ndarray, groups_row: np.ndarray,
                             folds_row: np.ndarray, ccabs_row: np.ndarray,
                             model_name: str, classes=CLASSES) -> dict:
    # Same as cv_run_long but each feature column is OLS-residualized on cc_abs
    # within the training fold before fitting.
    fold_scores = []
    for k in range(N_FOLDS):
        tr = folds_row != k; te = folds_row == k
        if tr.sum() == 0 or te.sum() == 0: continue
        # Per-feature OLS on cc_abs, training fold only.
        c_tr = ccabs_row[tr].reshape(-1, 1); c_te = ccabs_row[te].reshape(-1, 1)
        med = np.nanmedian(c_tr)
        c_tr_f = np.where(np.isnan(c_tr), med, c_tr)
        c_te_f = np.where(np.isnan(c_te), med, c_te)
        Xt = X[tr].copy(); Xe = X[te].copy()
        for j in range(Xt.shape[1]):
            f_tr = Xt[:, j]; m = ~np.isnan(f_tr)
            if m.sum() < 5: continue
            c_fit = c_tr_f[m, 0]; f_fit = f_tr[m]
            cm_, fm_ = c_fit.mean(), f_fit.mean()
            denom = ((c_fit - cm_) ** 2).sum()
            if denom < 1e-12: continue
            b = ((c_fit - cm_) * (f_fit - fm_)).sum() / denom
            a = fm_ - b * cm_
            Xt[:, j] = f_tr - (a + b * c_tr_f[:, 0])
            Xe[:, j] = Xe[:, j] - (a + b * c_te_f[:, 0])
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y_row[tr])
        pipe.fit(Xt, y_row[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y_row[te], proba, groups_row[te], pipe.classes_)
        fold_scores.append(sc)
    return summarize_cv_runs(fold_scores)


# Row-level arrays
y_row     = df['celltype_label'].to_numpy()
groups_r  = df['nucleus_id'].to_numpy()
folds_r   = df['gkf_fold_phase3'].to_numpy().astype(np.int8)
ccabs_r   = df['cc_abs'].to_numpy(dtype=np.float64)

print(f'long rows: {len(df):,}')
print('Helpers ready.')


long rows: 202,504
Helpers ready.


## 6. GKF main grid — 5 blocks × 2 models

In [6]:
results_rows = []
t0 = time.time()
for block_name, cols in BLOCKS.items():
    X = df[cols].to_numpy(dtype=np.float64)
    for model in ('LogReg', 'HGB'):
        s = cv_run_long(X, y_row, groups_r, folds_r, model)
        results_rows.append({
            'family': 'main', 'block': block_name, 'model': model,
            'cv': 'gkf', 'n_features': X.shape[1],
            'n_rows': int(X.shape[0]),
            **{k: v for k, v in s.items() if k != 'per_class_recall'},
            'recall_IT': s['per_class_recall'].get('5P-IT', float('nan')),
            'recall_ET': s['per_class_recall'].get('5P-ET', float('nan')),
        })
        print(f'[{block_name:<12s} | {model:<6s}] '
              f'bal_acc = {s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f} '
              f'| f1m = {s["macro_f1"]:.3f} '
              f'| R(IT)={s["per_class_recall"].get("5P-IT", float("nan")):.3f}, '
              f'R(ET)={s["per_class_recall"].get("5P-ET", float("nan")):.3f}')
print(f'\nGKF main grid done in {time.time()-t0:.1f}s')


[G            | LogReg] bal_acc = 0.678 ± 0.018 | f1m = 0.631 | R(IT)=0.709, R(ET)=0.647
[G            | HGB   ] bal_acc = 0.660 ± 0.025 | f1m = 0.682 | R(IT)=0.942, R(ET)=0.378
[A1+B         | LogReg] bal_acc = 0.670 ± 0.027 | f1m = 0.607 | R(IT)=0.653, R(ET)=0.687
[A1+B         | HGB   ] bal_acc = 0.684 ± 0.028 | f1m = 0.655 | R(IT)=0.771, R(ET)=0.597
[A1+B+C1      | LogReg] bal_acc = 0.640 ± 0.019 | f1m = 0.542 | R(IT)=0.518, R(ET)=0.761
[A1+B+C1      | HGB   ] bal_acc = 0.683 ± 0.034 | f1m = 0.689 | R(IT)=0.888, R(ET)=0.478
[A1+B+C1+D1   | LogReg] bal_acc = 0.650 ± 0.043 | f1m = 0.556 | R(IT)=0.538, R(ET)=0.762
[A1+B+C1+D1   | HGB   ] bal_acc = 0.719 ± 0.032 | f1m = 0.727 | R(IT)=0.903, R(ET)=0.534
[G+B+C1       | LogReg] bal_acc = 0.678 ± 0.018 | f1m = 0.631 | R(IT)=0.709, R(ET)=0.647
[G+B+C1       | HGB   ] bal_acc = 0.668 ± 0.023 | f1m = 0.690 | R(IT)=0.942, R(ET)=0.393

GKF main grid done in 151.8s


## 7. Confound baselines

In [7]:
from sklearn.metrics import balanced_accuracy_score, f1_score

# 7.1 Majority class (predict 5P-IT for everyone)
neuron_df = df.drop_duplicates('nucleus_id').reset_index(drop=True)
y_neuron = neuron_df['celltype_label'].to_numpy()
y_uniq, y_cnt = np.unique(y_neuron, return_counts=True)
maj_class = y_uniq[y_cnt.argmax()]
y_pred_maj = np.full_like(y_neuron, maj_class)
results_rows.append({
    'family': 'baseline', 'block': '-', 'model': 'majority', 'cv': 'gkf',
    'n_features': 0, 'n_rows': int(len(neuron_df)),
    'balanced_accuracy': float(balanced_accuracy_score(y_neuron, y_pred_maj)),
    'balanced_accuracy_std': 0.0,
    'macro_f1': float(f1_score(y_neuron, y_pred_maj, average='macro',
                               labels=CLASSES, zero_division=0)),
    'macro_f1_std': 0.0,
    'recall_IT': 1.0 if maj_class=='5P-IT' else 0.0,
    'recall_ET': 1.0 if maj_class=='5P-ET' else 0.0,
})
print(f'[baseline | majority    ] bal_acc = {results_rows[-1]["balanced_accuracy"]:.3f}')

# 7.2 Scan-only (one-hot of session_key on long rows; predict per-row, aggregate per neuron)
scan_dummies = pd.get_dummies(df['session_key'], prefix='scan').to_numpy(dtype=np.float64)
for model in ('LogReg', 'HGB'):
    s = cv_run_long(scan_dummies, y_row, groups_r, folds_r, model)
    results_rows.append({
        'family':'baseline','block':'scan-only','model':model,'cv':'gkf',
        'n_features': scan_dummies.shape[1],'n_rows':int(scan_dummies.shape[0]),
        **{k:v for k,v in s.items() if k!='per_class_recall'},
        'recall_IT': s['per_class_recall'].get('5P-IT', float('nan')),
        'recall_ET': s['per_class_recall'].get('5P-ET', float('nan')),
    })
    print(f'[baseline | scan-only   | {model:<6s}] bal_acc = '
          f'{s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f}')

# 7.3 cc_abs-only (single feature, broadcast per neuron, long-row protocol)
ccabs_X = df[['cc_abs']].to_numpy(dtype=np.float64)
for model in ('LogReg', 'HGB'):
    s = cv_run_long(ccabs_X, y_row, groups_r, folds_r, model)
    results_rows.append({
        'family':'baseline','block':'cc_abs-only','model':model,'cv':'gkf',
        'n_features':1,'n_rows':int(ccabs_X.shape[0]),
        **{k:v for k,v in s.items() if k!='per_class_recall'},
        'recall_IT': s['per_class_recall'].get('5P-IT', float('nan')),
        'recall_ET': s['per_class_recall'].get('5P-ET', float('nan')),
    })
    print(f'[baseline | cc_abs-only | {model:<6s}] bal_acc = '
          f'{s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f}')

# 7.4 cc_abs-residualized winner (decided after main grid)
main_df = pd.DataFrame([r for r in results_rows if r['family']=='main'])
winner = main_df.sort_values('balanced_accuracy', ascending=False).iloc[0]
WINNER_BLOCK = winner['block']; WINNER_MODEL = winner['model']
print(f'\nGKF winner: {WINNER_BLOCK} | {WINNER_MODEL} '
      f'(bal_acc = {winner["balanced_accuracy"]:.3f} ± {winner["balanced_accuracy_std"]:.3f})')

X_w = df[BLOCKS[WINNER_BLOCK]].to_numpy(dtype=np.float64)
s_resid = cv_run_long_residualized(X_w, y_row, groups_r, folds_r, ccabs_r, WINNER_MODEL)
results_rows.append({
    'family':'baseline','block':f'{WINNER_BLOCK} (resid cc_abs)','model':WINNER_MODEL,
    'cv':'gkf','n_features': len(BLOCKS[WINNER_BLOCK]),'n_rows':int(X_w.shape[0]),
    **{k:v for k,v in s_resid.items() if k!='per_class_recall'},
    'recall_IT': s_resid['per_class_recall'].get('5P-IT', float('nan')),
    'recall_ET': s_resid['per_class_recall'].get('5P-ET', float('nan')),
})
print(f'[{WINNER_BLOCK} | {WINNER_MODEL} | cc_abs-resid] bal_acc = '
      f'{s_resid["balanced_accuracy"]:.3f} ± {s_resid["balanced_accuracy_std"]:.3f}'
      f'  (Δ vs winner = {s_resid["balanced_accuracy"] - winner["balanced_accuracy"]:+.3f})')


[baseline | majority    ] bal_acc = 0.500
[baseline | scan-only   | LogReg] bal_acc = 0.731 ± 0.017
[baseline | scan-only   | HGB   ] bal_acc = 0.731 ± 0.017
[baseline | cc_abs-only | LogReg] bal_acc = 0.526 ± 0.028
[baseline | cc_abs-only | HGB   ] bal_acc = 0.527 ± 0.036

GKF winner: A1+B+C1+D1 | HGB (bal_acc = 0.719 ± 0.032)
[A1+B+C1+D1 | HGB | cc_abs-resid] bal_acc = 0.600 ± 0.032  (Δ vs winner = -0.119)


## 8. Headline summary

In [8]:
results = pd.DataFrame(results_rows)
results['bal_acc_str'] = results.apply(
    lambda r: f"{r['balanced_accuracy']:.3f} ± {r['balanced_accuracy_std']:.3f}", axis=1)

print('=== L5 IT/ET — GKF MAIN GRID (long-row + neuron-level prob aggregation) ===')
print(results[results['family']=='main']
      .sort_values('balanced_accuracy', ascending=False)
      [['block','model','n_features','bal_acc_str','recall_IT','recall_ET','macro_f1']]
      .to_string(index=False))
print()
print('=== L5 IT/ET — BASELINES ===')
print(results[results['family']=='baseline']
      [['block','model','n_features','bal_acc_str','recall_IT','recall_ET','macro_f1']]
      .to_string(index=False))


=== L5 IT/ET — GKF MAIN GRID (long-row + neuron-level prob aggregation) ===
     block  model  n_features   bal_acc_str  recall_IT  recall_ET  macro_f1
A1+B+C1+D1    HGB          41 0.719 ± 0.032   0.903297   0.534326  0.726724
      A1+B    HGB          27 0.684 ± 0.028   0.771379   0.597346  0.654755
   A1+B+C1    HGB          31 0.683 ± 0.034   0.887836   0.478104  0.689216
         G LogReg         116 0.678 ± 0.018   0.709268   0.646755  0.631169
    G+B+C1 LogReg         127 0.678 ± 0.018   0.709268   0.646755  0.631169
      A1+B LogReg          27 0.670 ± 0.027   0.653322   0.686737  0.606704
    G+B+C1    HGB         127 0.668 ± 0.023   0.941954   0.393405  0.690341
         G    HGB         116 0.660 ± 0.025   0.941910   0.377796  0.682449
A1+B+C1+D1 LogReg          41 0.650 ± 0.043   0.537797   0.761561  0.555915
   A1+B+C1 LogReg          31 0.640 ± 0.019   0.518028   0.761119  0.541775

=== L5 IT/ET — BASELINES ===
                    block    model  n_features   bal_acc_s

## 9. Save

In [9]:
results.to_parquet(RESULTS_OUT, index=False)
print(f'wrote {RESULTS_OUT}  ({RESULTS_OUT.stat().st_size/1024:.1f} KB, {len(results)} rows)')

winner_meta = {
    'phase':       'phase3_step2_v2_longrow',
    'task':        'L5_IT_vs_ET',
    'protocol':    'long-row + per-neuron probability aggregation (WORKFLOW §3.6)',
    'winner_block': WINNER_BLOCK,
    'winner_model': WINNER_MODEL,
    'winner_balanced_accuracy_mean': float(winner['balanced_accuracy']),
    'winner_balanced_accuracy_std':  float(winner['balanced_accuracy_std']),
    'cc_abs_resid_balanced_accuracy': float(s_resid['balanced_accuracy']),
    'cc_abs_resid_delta':            float(s_resid['balanced_accuracy'] - winner['balanced_accuracy']),
    'n_long_rows': int(len(df)),
    'n_neurons':   int(df['nucleus_id'].nunique()),
}
with open(WINNER_OUT, 'w') as f:
    json.dump(winner_meta, f, indent=2)
print(f'wrote {WINNER_OUT}')


wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l5_it_et_runs.parquet  (9.3 KB, 16 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l5_it_et_winner.json


## 10. Step-2 summary

In [10]:
print('=== PHASE 3 STEP 2 (REBUILD) SUMMARY ===')
print(f'protocol          : long-row + per-neuron probability aggregation (WORKFLOW §3.6)')
print(f'long rows         : {len(df):,}')
print(f'neurons (5P-IT)   : {int((y_neuron=="5P-IT").sum())}')
print(f'neurons (5P-ET)   : {int((y_neuron=="5P-ET").sum())}')
print()
print(f'GKF winner        : {WINNER_BLOCK} | {WINNER_MODEL}')
print(f'  bal_acc          = {winner["balanced_accuracy"]:.3f} ± {winner["balanced_accuracy_std"]:.3f}')
print(f'  cc_abs-resid     = {s_resid["balanced_accuracy"]:.3f} '
      f'± {s_resid["balanced_accuracy_std"]:.3f}  '
      f'(Δ = {s_resid["balanced_accuracy"]-winner["balanced_accuracy"]:+.3f})')
print()
maj_row = results.loc[results["model"]=="majority"].iloc[0]
ca_lr = results.loc[(results["block"]=="cc_abs-only") & (results["model"]=="LogReg")].iloc[0]
ca_hgb = results.loc[(results["block"]=="cc_abs-only") & (results["model"]=="HGB")].iloc[0]
sc_lr = results.loc[(results["block"]=="scan-only") & (results["model"]=="LogReg")].iloc[0]
sc_hgb = results.loc[(results["block"]=="scan-only") & (results["model"]=="HGB")].iloc[0]
print(f'baselines        : majority={maj_row["balanced_accuracy"]:.3f}, '
      f'scan LR={sc_lr["balanced_accuracy"]:.3f}, scan HGB={sc_hgb["balanced_accuracy"]:.3f}, '
      f'cc_abs LR={ca_lr["balanced_accuracy"]:.3f}, cc_abs HGB={ca_hgb["balanced_accuracy"]:.3f}')


=== PHASE 3 STEP 2 (REBUILD) SUMMARY ===
protocol          : long-row + per-neuron probability aggregation (WORKFLOW §3.6)
long rows         : 202,504
neurons (5P-IT)   : 1169
neurons (5P-ET)   : 320

GKF winner        : A1+B+C1+D1 | HGB
  bal_acc          = 0.719 ± 0.032
  cc_abs-resid     = 0.600 ± 0.032  (Δ = -0.119)

baselines        : majority=0.500, scan LR=0.731, scan HGB=0.731, cc_abs LR=0.526, cc_abs HGB=0.527
